In [9]:
import numpy as np
import pandas as pd

data = pd.read_csv('PlayerStatistics.csv', low_memory=False)

# Isolate seasons & game type
data['gameDate'] = pd.to_datetime(data['gameDate'])
data = data.loc[(data['gameDate'] > '1996-09-01')&(data['gameType'].isin(['Regular Season', 'NBA Cup', 'NBA Emirates Cup']))]

# Compute team winrates per season
team_wins = data[['gameId', 'playerteamName', 'win', 'gameDate']].drop_duplicates().reset_index(drop=True)
team_wins['season'] = team_wins['gameDate'].dt.year.where(team_wins['gameDate'].dt.month >= 10,
                                                      team_wins['gameDate'].dt.year - 1) + 1
tw_pct = team_wins.groupby(['season', 'playerteamName'])['win'].mean().reset_index(name='win_pct')

# Isolate players that played any minutes
data = data.loc[~data['numMinutes'].isna()]

# Isolate players with a (minimal) substantial number of games
# Mostly so that there are few missing values when matching players with birth dates
data['pName'] = data['firstName'] + ' ' + data['lastName']

counts = data.groupby(['pName', 'personId']).size().reset_index(name='count')
valid_pairs = counts.query('count >= 200')[['pName', 'personId']]
data = data.merge(valid_pairs, on=['pName','personId'])

# +- 1 million rows truncated in this cell

In [10]:
# Isolate games played in ages 25-31
bio_data = pd.read_csv('Players.csv', low_memory=False)
bio_data['birthdate'] = pd.to_datetime(bio_data['birthdate'])

data = data.merge(bio_data[['personId', 'birthdate']], on='personId', how='left')

# Keep games for seasons where players were 25-31 at some point
data['season_start_year'] = data['gameDate'].dt.year.where(data['gameDate'].dt.month >= 10,
                                                      data['gameDate'].dt.year - 1)
data['season'] = data['season_start_year']+1
data['season_start'] = pd.to_datetime(data['season_start_year'].astype(str) + '-10-01')
data['season_end'] = pd.to_datetime((data['season_start_year'] + 1).astype(str) + '-06-30')

data['age_start'] = (data['season_start'] - data['birthdate']).dt.days // 365
data['age_end'] = (data['season_end'] - data['birthdate']).dt.days // 365

data = data[(data['age_start']<=31)&(data['age_end']>=24)]
data['age'] = data['age_end']
data = data.drop(['season_start_year', 'season_start', 'season_end', 'age_start', 'age_end'], axis=1)

# Isolate players with over 200 games in the stretch
counts = data.groupby(['pName', 'personId']).size().reset_index(name='count')
valid_pairs = counts.query('count >= 200')[['pName', 'personId']]
data = data.merge(valid_pairs, on=['pName','personId'])

data = data.reset_index(drop=True)

# +- 250 thousand rows truncated in this cell
# Result:
# Seasons: 1996/97 - 2024/25
# Player Ages: 24-31 (at any point in the season for season completeness)
# Total game sample per player: >200 games
# Game type: Regular Season

### Start Data Construction

In [11]:
# Firstly, team-wise stats (for posseessions for example)
# Also same trimming as in core data

totals = pd.read_csv('PlayerStatistics.csv', low_memory=False)

# Isolate seasons & game type
totals['gameDate'] = pd.to_datetime(totals['gameDate'])
totals = totals.loc[(totals['gameDate'] > '1996-09-01')&(totals['gameType'].isin(['Regular Season', 'NBA Cup', 'NBA Emirates Cup']))]

totals['pName'] = totals['firstName'] + ' ' + totals['lastName']
totals['season'] = totals['gameDate'].dt.year.where(totals['gameDate'].dt.month >= 10, totals['gameDate'].dt.year - 1) + 1

# Aggregating on numerical columns
totals_agg = totals.groupby(['playerteamName', 'season'])[totals.columns.to_list()[15:-2]].agg('sum').reset_index()
totals_agg.columns = ['playerteamName', 'season'] + (totals_agg.columns[2:]+'Team').tolist()

# Need to divide minutes by 5 for 5 players on the floor
totals_agg['numMinutesTeam'] = totals_agg['numMinutesTeam'] / 5

# Compute possessions per team per season
# As: TeamPossessions = FGA + 0.44 * FTA - ORB + TOV
totals_agg['posTeam'] = totals_agg['fieldGoalsAttemptedTeam'] + 0.44 * totals_agg['freeThrowsAttemptedTeam'] - totals_agg['reboundsOffensiveTeam'] + totals_agg['turnoversTeam']

In [12]:
# Data Construction
base = data.groupby(['personId', 'pName', 'season', 'playerteamName', 'age'])['points'].agg('count').reset_index()
base.rename(columns={'points': 'GP'}, inplace=True)

base = base.merge(bio_data[['personId', 'height', 'bodyWeight']], on=['personId'] , how='left')
base = base.merge(tw_pct, on=['season', 'playerteamName'], how='left')

# Keep only player seasons with single team
team_counts = base.groupby(['personId', 'season'])['playerteamName'].nunique().reset_index()
team_counts = team_counts.rename(columns={'playerteamName': 'team_count'})
single_team = team_counts[team_counts['team_count'] == 1]

base = base.merge(single_team[['personId', 'season']], on=['personId', 'season'], how='right')

# Adding raw stats
raw_stats = ['numMinutes', 'points', 'assists', 'blocks', 'steals', 'fieldGoalsAttempted', 'fieldGoalsMade',
             'threePointersAttempted', 'threePointersMade', 'freeThrowsAttempted', 'freeThrowsMade', 'reboundsDefensive',
             'reboundsOffensive', 'foulsPersonal', 'turnovers', 'plusMinusPoints']
             
base = base.merge(data.groupby(['personId', 'season'])[raw_stats].agg('sum').reset_index(), on=['personId', 'season'], how='left')

# Remove players with insignificant sample size per season (at least 30 games played)
base = base.loc[base['GP']>29].reset_index(drop=True)

# Convert all metrics from totals to per-100 possessions
# Merge with team stats
base = base.merge(totals_agg[['season', 'playerteamName', 'posTeam', 'numMinutesTeam']], on=['season', 'playerteamName'], how='left')

# Convert all stats to per-100 pos
for stat in base.columns[10:-2]:
    base[f'{stat}_per100'] = 100 * base[stat] / base['posTeam'] * (base['numMinutesTeam'] / base['numMinutes'])

### Data Construction Continued

In [84]:
# Effective field goal percentage (for dimensionality reduction)
base['eFG'] = (base['fieldGoalsMade'] + 0.5 * base['threePointersMade']) / base['fieldGoalsAttempted']

# Total rebound calculation
base['rebounds'] = base['reboundsOffensive'] + base['reboundsDefensive']
base['rebounds_per100'] = base['reboundsOffensive_per100'] + base['reboundsDefensive_per100']

# Assist-to-Turnover ratio
# Assists and turnovers can simply be a function of having the ball more in one's hands, thus correlating these 2 metrics
# The metric is not without flaws but is still good for variable decoupling
base['astTov'] = base['assists'] / base['turnovers']
base['astTov_per100'] = base['assists'] / base['turnovers'] # Not different than astTov since both are ratios but needed for the next line of code

# Decoupling example
display(base[[i+'_per100' for i in ['points', 'rebounds', 'astTov', 'plusMinusPoints']]].corr())
display(base[[i+'_per100' for i in ['points', 'rebounds', 'assists', 'turnovers', 'plusMinusPoints']]].corr())

,points_per100,rebounds_per100,astTov_per100,plusMinusPoints_per100
points_per100,1.000000,-0.026322,-0.003971,0.237877
rebounds_per100,-0.026322,1.000000,-0.550676,0.056592
astTov_per100,-0.003971,-0.550676,1.000000,0.169403
plusMinusPoints_per100,0.237877,0.056592,0.169403,1.000000


,points_per100,rebounds_per100,assists_per100,turnovers_per100,plusMinusPoints_per100
points_per100,1.000000,-0.026322,0.259309,0.458110,0.237877
rebounds_per100,-0.026322,1.000000,-0.449896,0.025512,0.056592
assists_per100,0.259309,-0.449896,1.000000,0.550201,0.139748
turnovers_per100,0.458110,0.025512,0.550201,1.000000,-0.006583
plusMinusPoints_per100,0.237877,0.056592,0.139748,-0.006583,1.000000


In [85]:
# Get number of possessions per game so that we can get per100pos stats for each game and compute
# standard deviations from it

pos_per_game = totals.groupby(['gameId', 'playerteamName'])[['fieldGoalsAttempted', 'freeThrowsAttempted', 'reboundsOffensive', 'turnovers', 'numMinutes']].agg('sum').reset_index()
pos_per_game['posTeam'] = pos_per_game['fieldGoalsAttempted'] + 0.44*pos_per_game['freeThrowsAttempted'] - pos_per_game['reboundsOffensive'] + pos_per_game['turnovers']
pos_per_game.rename(columns={'numMinutes': 'numMinutesTeam'}, inplace=True)
pos_per_game['numMinutesTeam'] == pos_per_game['numMinutesTeam'] / 5

# Append individual game possessions
data = pd.merge(data, pos_per_game[['gameId', 'playerteamName', 'numMinutesTeam', 'posTeam']], on=['gameId', 'playerteamName'], how='left')

# Convert all stats to per-100 pos
for stat in data.columns[16:-6]:
    data[f'{stat}_per100'] = 100 * data[stat] / data['posTeam'] * (data['numMinutesTeam'] / 5) / data['numMinutes']

# Removing irrelevant columns
data.drop(columns=['gameLabel', 'gameSubLabel', 'seriesGameNumber'], inplace=True)

In [86]:
# Resillience

# The data is currently sorted for date loosely (for a single game, there is an index mix of 2 teams)
# This will be strictly fixed now for future use
data = data.sort_values(['season', 'gameDate', 'gameId', 'playerteamName', 'firstName', 'lastName'], ascending=False)

# Mirroring the mean stats in the base dataset
data['astTov'] = data['assists'] / data['turnovers']
data['eFG'] = (data['fieldGoalsMade'] + 0.5 * data['threePointersMade']) / data['fieldGoalsAttempted']
data['eFG'] = data['eFG'].fillna(0)
data['rebounds'] = data['reboundsOffensive'] + data['reboundsDefensive']
data['rebounds_per100'] = data['reboundsOffensive_per100'] + data['reboundsDefensive_per100']

# Fill na for per 100 stats that were divided by 0
data = data.replace([np.inf, -np.inf], 0).fillna(0)

In [87]:
# Core statistics' standard deviations

# Drop irrelevant games that are inflating the standard deviations
data = data.loc[data['numMinutes']>5].reset_index(drop=False)

# Handle NaNs and infitite values in the AST-to-TOV ratio column
def safe_std(x):
    x = x[np.isfinite(x)]
    return x.std(ddof=0) if len(x) > 0 else np.nan

data_std = data.groupby(['personId', 'season'])[['points_per100', 'rebounds_per100', 'astTov', 'eFG', 'plusMinusPoints_per100']].agg(safe_std).reset_index()
data_std.columns = ['personId', 'season'] + (data_std.columns[2:]+'Std').tolist()

base = base.merge(data_std, on=['personId', 'season'], how='left')

In [90]:
# Leadership

# "AST% is an estimate of the percentage of teammate field goals a player assisted while he was on the floor"
# AST / (((MP / (Tm MP / 5)) * Tm FG) - FG)
base = pd.merge(base, totals_agg[['playerteamName', 'season', 'fieldGoalsMadeTeam']], on=['playerteamName', 'season'], how='left')
base['AST%'] = base['assists'] / (((base['numMinutes'] / base['numMinutesTeam']) * base['fieldGoalsMadeTeam']) - base['fieldGoalsMade'])

In [91]:
# Adaptability

# Steals + Blocks indicating ability to guard both "big and small"
base['stocks'] = base['steals'] + base['blocks']
base['stocks_per100'] = base['steals_per100'] + base['blocks_per100']

In [92]:
# Work engagement - Already contained

### Player Salaries

In [93]:
salaries = pd.read_csv('Salaries.csv')
salaries.columns = ['pName', 'season', 'salary']
salaries['season'] = salaries['season'] + 1 # Since this dataset counts seasons differently
salaries['salary'] = pd.to_numeric(salaries['salary'].str.replace(',', ''), errors='coerce') # Str format to numeric

base = base.merge(salaries, on=['pName', 'season'], how='left')

# Deflate the salaries and adjust them for salary cap
cap = pd.read_excel('Cap.xlsx')

base = base.merge(cap, on='season', how='left')
base['salary_adj'] = base['salary'] / base['cap']

### MPG

In [94]:
base['MPG'] = base['numMinutes'] / base['GP']

### Export

In [95]:
base.to_csv('Data01.csv', index=False)